# Rank Classifier Training (MobileNetV3-Small)

Train a MobileNetV3-Small model to classify playing card **ranks** (13 classes: A, 2–10, J, Q, K).

This is Stage 2 of the 24 Game Solver ML pipeline:
- **Stage 1** (YOLO): Detect and crop individual cards from a camera frame
- **Stage 2** (this notebook): Classify the cropped card image into one of 13 ranks

The trained model is exported to **TFLite FP16** for on-device inference in the Flutter app.

> **Runtime**: GPU recommended. In Colab: Runtime → Change runtime type → T4 GPU.

## 1. Setup

In [ ]:
# Install dependencies
!pip install -q tensorflow matplotlib seaborn kaggle scikit-learn pyyaml Pillow

import os
import shutil
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

## 2. Download & Organize Dataset

We reuse the same [Playing Cards Object Detection Dataset](https://www.kaggle.com/datasets/andy8744/playing-cards-object-detection-dataset) from Notebook 01. The dataset has 52 per-card classes (e.g. `ace of clubs`, `king of hearts`). We:

1. Download the dataset (if not already present from Notebook 01)
2. Crop each card using its YOLO bounding box annotation
3. Organize crops into 13 rank-only folders (A, 2–10, J, Q, K)

**Before running:** Set your Kaggle API token:
```python
import os
os.environ["KAGGLE_API_TOKEN"] = "your_token_here"
```

In [ ]:
# Download the dataset (skip if already downloaded in Notebook 01)
DATA_DIR = "data"
if not os.path.isdir(os.path.join(DATA_DIR, "train")):
    !kaggle datasets download -d andy8744/playing-cards-object-detection-dataset -p {DATA_DIR} --unzip
    print("Dataset downloaded.")
else:
    print("Dataset already present, skipping download.")

# Read the original data.yaml to get class name → index mapping
# The dataset ships with a data.yaml listing all 52 classes
import yaml

original_yaml = os.path.join(DATA_DIR, "data.yaml")
with open(original_yaml) as f:
    ds_config = yaml.safe_load(f)

# ds_config["names"] maps class_id → class_name (e.g. {0: "ace of clubs", ...})
class_names_52 = ds_config.get("names", {})
print(f"Found {len(class_names_52)} classes in dataset")
for i in list(class_names_52.items())[:5]:
    print(f"  {i}")

# Build class_id → rank mapping
RANK_MAPPING = {
    "ace": "A", "two": "2", "three": "3", "four": "4", "five": "5",
    "six": "6", "seven": "7", "eight": "8", "nine": "9", "ten": "10",
    "jack": "J", "queen": "Q", "king": "K",
    "2": "2", "3": "3", "4": "4", "5": "5", "6": "6",
    "7": "7", "8": "8", "9": "9", "10": "10",
}

def class_name_to_rank(name: str) -> str:
    """Extract rank from a class name like 'ace of clubs' or '10 of hearts'."""
    name_lower = name.lower()
    for key, rank in RANK_MAPPING.items():
        if key in name_lower:
            return rank
    return None

id_to_rank = {}
for cid, cname in class_names_52.items():
    rank = class_name_to_rank(str(cname))
    if rank:
        id_to_rank[int(cid)] = rank
    else:
        print(f"  WARNING: Could not map class {cid}: '{cname}'")

print(f"\nMapped {len(id_to_rank)} class IDs to ranks")
print(f"Ranks: {sorted(set(id_to_rank.values()))}")

### Crop Cards from Bounding Boxes

Use the YOLO annotations to crop individual cards from each image, then save them into rank-based folders. This creates a clean classification dataset from the detection dataset.

In [ ]:
# Crop cards from images using YOLO bounding boxes and organize by rank
from PIL import Image

RANK_DIR = "data/rank_cards"
os.makedirs(RANK_DIR, exist_ok=True)
for rank in ["A", "2", "3", "4", "5", "6", "7", "8", "9", "10", "J", "Q", "K"]:
    os.makedirs(os.path.join(RANK_DIR, rank), exist_ok=True)

total_crops = 0

for split in ["train", "valid", "test"]:
    img_dir = os.path.join(DATA_DIR, split, "images")
    lbl_dir = os.path.join(DATA_DIR, split, "labels")
    if not os.path.isdir(img_dir):
        continue

    for lbl_file in sorted(os.listdir(lbl_dir)):
        if not lbl_file.endswith(".txt"):
            continue

        # Find corresponding image
        stem = Path(lbl_file).stem
        img_path = None
        for ext in [".jpg", ".jpeg", ".png"]:
            candidate = os.path.join(img_dir, stem + ext)
            if os.path.exists(candidate):
                img_path = candidate
                break
        if img_path is None:
            continue

        img = Image.open(img_path)
        w, h = img.size

        with open(os.path.join(lbl_dir, lbl_file)) as f:
            for line_idx, line in enumerate(f):
                parts = line.strip().split()
                if len(parts) < 5:
                    continue

                class_id = int(parts[0])
                rank = id_to_rank.get(class_id)
                if rank is None:
                    continue

                # YOLO format: cx, cy, bw, bh (normalized)
                cx, cy, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                x1 = int((cx - bw / 2) * w)
                y1 = int((cy - bh / 2) * h)
                x2 = int((cx + bw / 2) * w)
                y2 = int((cy + bh / 2) * h)

                # Clamp to image bounds
                x1, y1 = max(0, x1), max(0, y1)
                x2, y2 = min(w, x2), min(h, y2)

                if x2 - x1 < 10 or y2 - y1 < 10:
                    continue

                crop = img.crop((x1, y1, x2, y2))
                out_path = os.path.join(RANK_DIR, rank, f"{split}_{stem}_{line_idx}.jpg")
                crop.save(out_path, quality=95)
                total_crops += 1

print(f"\nCropped {total_crops} card images into rank folders:")
for rank in sorted(os.listdir(RANK_DIR)):
    count = len(os.listdir(os.path.join(RANK_DIR, rank)))
    print(f"  {rank}: {count} images")

In [ ]:
IMG_SIZE = 128
BATCH_SIZE = 32
RANK_DIR = "data/rank_cards"

# Class names in sorted order — this determines the index→rank mapping
CLASS_NAMES = sorted(os.listdir(RANK_DIR))
print(f"Classes ({len(CLASS_NAMES)}): {CLASS_NAMES}")
assert len(CLASS_NAMES) == 13, f"Expected 13 classes, got {len(CLASS_NAMES)}"

# Load datasets with augmentation for training
train_ds = tf.keras.utils.image_dataset_from_directory(
    RANK_DIR,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="int",
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    RANK_DIR,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="int",
)

# Normalize to [0, 1]
normalization = tf.keras.layers.Rescaling(1.0 / 255)
train_ds = train_ds.map(lambda x, y: (normalization(x), y))
val_ds = val_ds.map(lambda x, y: (normalization(x), y))

# Data augmentation (applied only during training)
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.04),          # ±15 degrees
    tf.keras.layers.RandomBrightness(0.2),
    tf.keras.layers.RandomContrast(0.2),
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomZoom(0.1),
])

train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y))

# Prefetch for performance
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

print(f"Training batches: {tf.data.experimental.cardinality(train_ds).numpy()}")
print(f"Validation batches: {tf.data.experimental.cardinality(val_ds).numpy()}")

## 4. Build Model

MobileNetV3-Small pretrained on ImageNet, with a custom classification head for 13 rank classes. The backbone is frozen initially and then unfrozen for fine-tuning.

In [ ]:
# MobileNetV3-Small with custom classification head
base = tf.keras.applications.MobileNetV3Small(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet",
)
base.trainable = False  # freeze backbone initially

model = tf.keras.Sequential([
    base,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(13, activation="softmax"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()
print(f"\nTotal params: {model.count_params():,}")

## 5. Train Phase 1: Frozen Backbone

Train only the classification head for 5 epochs. The ImageNet backbone weights are frozen — this quickly teaches the head to map MobileNetV3 features to card ranks.

In [ ]:
# Phase 1: Train only the classification head (backbone frozen)
history_frozen = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
)

print(f"Phase 1 val accuracy: {history_frozen.history['val_accuracy'][-1]:.4f}")

## 6. Train Phase 2: Fine-tune

Unfreeze the full backbone and fine-tune at a lower learning rate (1e-4). Early stopping monitors validation accuracy with patience=5 to avoid overfitting.

In [ ]:
# Phase 2: Unfreeze backbone and fine-tune at lower learning rate
base.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history_finetune = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_accuracy", patience=5, restore_best_weights=True
        ),
    ],
)

print(f"Phase 2 best val accuracy: {max(history_finetune.history['val_accuracy']):.4f}")

## 7. Evaluate: Confusion Matrix

Visualize per-class performance and flag systematic confusions (e.g., 6 vs 9, J/Q/K face cards). The confusion matrix is saved to `outputs/classifier/confusion_matrix.png`.

In [ ]:
# Confusion matrix to catch systematic errors (6/9, J/Q/K)
from sklearn.metrics import confusion_matrix, classification_report

y_true = []
y_pred = []

for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

cm = confusion_matrix(y_true, y_pred)
accuracy = np.mean(y_true == y_pred)
print(f"Overall accuracy: {accuracy:.4f}")
print(f"\nPer-class report:\n")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title(f"Rank Classifier Confusion Matrix (accuracy: {accuracy:.2%})")
plt.tight_layout()
os.makedirs("outputs/classifier", exist_ok=True)
plt.savefig("outputs/classifier/confusion_matrix.png", dpi=150)
plt.show()

# Flag common confusions
for i in range(len(CLASS_NAMES)):
    for j in range(len(CLASS_NAMES)):
        if i != j and cm[i][j] > 2:
            print(f"  WARNING: {CLASS_NAMES[i]} confused with {CLASS_NAMES[j]} ({cm[i][j]} times)")

## 8. Export to TFLite

Convert the trained Keras model to TFLite with FP16 quantization. FP16 halves the model size with negligible accuracy loss, making it suitable for on-device inference.

Output files:
- `outputs/classifier/rank_classifier.keras` — full Keras model (for resuming training)
- `outputs/classifier/rank_classifier.tflite` — quantized TFLite model (for Flutter app)
- `outputs/classifier/class_names.json` — index-to-rank mapping

In [ ]:
# Export to TFLite with FP16 quantization
os.makedirs("outputs/classifier", exist_ok=True)
model.save("outputs/classifier/rank_classifier.keras")

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_model = converter.convert()

tflite_path = "outputs/classifier/rank_classifier.tflite"
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

size_mb = os.path.getsize(tflite_path) / (1024 * 1024)
print(f"TFLite model saved: {tflite_path} ({size_mb:.1f} MB)")

# Save class name mapping for reference
import json
class_map = {i: name for i, name in enumerate(CLASS_NAMES)}
with open("outputs/classifier/class_names.json", "w") as f:
    json.dump(class_map, f, indent=2)
print(f"Class mapping: {class_map}")
print("\nCopy rank_classifier.tflite to flutter_app/assets/models/rank_classifier.tflite")

## 9. Verify TFLite Model

Run a quick sanity check by loading the TFLite model and running inference on a few validation images. This confirms the exported model produces correct predictions before deploying to the Flutter app.

In [ ]:
# Quick sanity check: run TFLite model on a few validation images
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(f"Input:  shape={input_details[0]['shape']}, dtype={input_details[0]['dtype']}")
print(f"Output: shape={output_details[0]['shape']}, dtype={output_details[0]['dtype']}")

# Test on 5 validation images
correct = 0
total = 0
for images, labels in val_ds.take(1):
    for i in range(min(5, len(images))):
        img = images[i:i+1].numpy().astype(np.float32)
        interpreter.set_tensor(input_details[0]["index"], img)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details[0]["index"])
        pred_class = np.argmax(output[0])
        true_class = labels[i].numpy()
        match = "✓" if pred_class == true_class else "✗"
        print(f"  {match} True: {CLASS_NAMES[true_class]}, Predicted: {CLASS_NAMES[pred_class]} "
              f"(conf: {output[0][pred_class]:.2f})")
        if pred_class == true_class:
            correct += 1
        total += 1

print(f"\nTFLite quick check: {correct}/{total} correct")